# OCT Classification

This first run CNN will use Kaggle's OCT-C8 Classification Dataset along with Torch's ResNet50 to act as a model base. We will train using C8, and then once we acquire barcoding data, we will tune it on that. 

This notebook is for sanity checking the folder structure, verifying image counts, and then looking at image dimensions. We will also take a look at samples for possible hypertransmission or other important aspects, and then identify needs for preprocessing

In [ ]:
# Imports
from pathlib import Path
from PIL import Image

import random
import pandas as pd
import matplotlib.pyplot as plt

# Locate Dataset
# DATA_DIR = Path("../data/oct_c8/RetinalOCT_Dataset/RetinalOCT_Dataset") # mac
DATA_DIR = Path("..\data\RetinalOCT_Dataset") #win


print(DATA_DIR.exists())
print(DATA_DIR)
print("\n")
for split_dir in DATA_DIR.iterdir():
    if split_dir.is_dir():
        print(split_dir.name)
        for class_dir in split_dir.iterdir():
            if class_dir.is_dir():
                print("  ", class_dir.name)

In [ ]:
splits = ["train", "val", "test"]
rows = []

for split in splits:
    split_dir = DATA_DIR / split
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            image_files = list(class_dir.glob("*.jpg"))
            rows.append({
                "split": split,
                "class": class_dir.name,
                "count": len(image_files)
            })

counts_df = pd.DataFrame(rows)

counts_pivot = counts_df.pivot(
    index="class",
    columns="split",
    values="count"
)

counts_pivot["total"] = counts_pivot.sum(axis=1)

counts_pivot

In [ ]:
image_info = []

for split in splits:
    split_dir = DATA_DIR / split
    
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            image_files = list(class_dir.glob("*.jpg"))
            
            sample_files = random.sample(
                image_files,
                min(50, len(image_files))
            )
            
            for img_path in sample_files:
                with Image.open(img_path) as img:
                    image_info.append({
                        "split": split,
                        "class": class_dir.name,
                        "width": img.width,
                        "height": img.height,
                        "mode": img.mode
                    })

image_info_df = pd.DataFrame(image_info)

image_info_df.groupby("split")[["width", "height"]].describe()

In [ ]:
image_info_df["mode"].value_counts()

In [ ]:
train_dir = DATA_DIR / "train"

classes = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])

fig, axes = plt.subplots(
    len(classes),
    4,
    figsize=(12, 2.5 * len(classes))
)

for i, cls in enumerate(classes):
    class_dir = train_dir / cls
    image_files = list(class_dir.glob("*.jpg"))
    sample_files = random.sample(image_files, 4)
    
    for j, img_path in enumerate(sample_files):
        img = Image.open(img_path)
        
        axes[i, j].imshow(img, cmap="gray")
        axes[i, j].set_title(cls)
        axes[i, j].axis("off")

plt.tight_layout()
plt.show()

The dataset has 24,000 OCT images (2300 training, 350 validation, 350 test) and is perfectly balanced

Image height has been standardized but widths are ranged, most images falling into 512-768 pixels. For this first run, we'll scale to 224x224, keeping in mind more preprocessing can be explored if anatomical dimensions become important

The dataset has both RGB (3-channel) and L (single channel grayscale) images. Since pretrained architectures expect RGB inputs, we will convert all to RGB during preprocessing

In [ ]:
# from torchvision import transforms

# transform = transforms.Compose([
#     transforms.Lambda(lambda x: x.convert("RGB")),
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
# ])

# Visual Inspection

Representative examples from each disease category demonstrate substantial variation in retinal morphology, image contrast, acquisition settings, and field of view.

Several AMD-, CNV-, and DRUSEN-associated scans exhibit regions of increased posterior signal transmission extending into the choroid. These regions are visually similar to the hypertransmission phenomena that motivated the barcoding project.

Although this dataset does not contain explicit barcoding annotations, it provides a useful benchmark for developing and validating a CNN-based OCT classification pipeline. In particular, it will allow evaluation of transfer learning strategies and Grad-CAM explainability methods before adapting the framework to barcoding-specific data.

The next phase will train a pretrained ResNet50 classifier on the eight retinal disease categories and assess whether Grad-CAM localization identifies clinically meaningful retinal structures.